In [ ]:
from google.colab import files
uploaded = files.upload()

Saving BikeNonbike.zip to BikeNonbike.zip


In [ ]:
import zipfile
import os
import cv2
import h5py
import numpy as np

In [ ]:
with zipfile.ZipFile("BikeNonbike.zip", 'r') as zip_ref:
    zip_ref.extractall()

print("Dataset extracted")

Dataset extracted


In [ ]:
print(os.listdir("BikeNonbike"))

['Nonbike', 'Bike']


In [ ]:
IMG_SIZE = 128

images = []
labels = []

BIKE_FOLDER = "BikeNonbike/Bike"
NON_BIKE_FOLDER = "BikeNonbike/Nonbike"

for file in os.listdir(BIKE_FOLDER):

    path = os.path.join(BIKE_FOLDER, file)

    img = cv2.imread(path)

    if img is None:
        continue

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)
    labels.append(1)


for file in os.listdir(NON_BIKE_FOLDER):

    path = os.path.join(NON_BIKE_FOLDER, file)

    img = cv2.imread(path)

    if img is None:
        continue

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)
    labels.append(0)

X = np.array(images, dtype=np.float32)
Y = np.array(labels)

X = X / 255.0

print("X shape:", X.shape)
print("Y shape:", Y.shape)

with h5py.File("data.h5", "w") as hf:

    hf.create_dataset("X", data=X)
    hf.create_dataset("Y", data=Y)

print("data.h5 created successfully")

X shape: (200, 128, 128, 3)
Y shape: (200,)
data.h5 created successfully


In [ ]:
with h5py.File("data.h5", "r") as hf:

    X = hf["X"][:]
    Y = hf["Y"][:]

print(X.shape)
print(Y.shape)

(200, 128, 128, 3)
(200,)


In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [ ]:
from sklearn.utils import shuffle
X, Y = shuffle(X, Y, random_state=42)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.5250 - loss: 1.0861 - val_accuracy: 0.6500 - val_loss: 0.5606
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.7875 - loss: 0.5062 - val_accuracy: 0.8000 - val_loss: 0.4010
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 999ms/step - accuracy: 0.8438 - loss: 0.3433 - val_accuracy: 0.8250 - val_loss: 0.3397
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 952ms/step - accuracy: 0.9000 - loss: 0.2488 - val_accuracy: 0.8250 - val_loss: 0.5313
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.9500 - loss: 0.1702 - val_accuracy: 0.8500 - val_loss: 0.2608
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.9438 - loss: 0.1499 - val_accuracy: 0.9500 - val_loss: 0.1798
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.9312 - loss: 0.1225 - val_accuracy: 0.9500 - val_loss: 0.1869
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 925ms/step - accuracy: 0.9438 - loss: 0.1214 - val_accuracy: 0.9250 - val_loss: 0.1889
Epoch 9

In [ ]:
model.evaluate(x_test, y_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 117ms/step - accuracy: 0.9750 - loss: 0.0999


[0.09992165863513947, 0.9750000238418579]

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
model.save("bike_classifier.h5")
print("Model saved")

Model saved


In [ ]:

uploaded = files.upload()

filename = list(uploaded.keys())[0]

model = load_model("bike_classifier.h5")

img = cv2.imread(filename)

img = cv2.resize(img, (128, 128))

img = img.astype(np.float32) / 255.0

img = np.expand_dims(img, axis=0)

print("Input shape:", img.shape)

prediction = model.predict(img)

print("Raw Prediction:", prediction[0][0])

if prediction[0][0] >= 0.5:
    print("Prediction: BIKE")
else:
    print("Prediction: NON-BIKE")

Saving 0.jpg to 0 (2).jpg
Input shape: (1, 128, 128, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
Raw Prediction: 0.99745977
Prediction: BIKE
